In [1]:
import pandas as pd

Load all datasets

In [2]:
customers = pd.read_csv("raw_data/olist_customers_dataset.csv")
orders = pd.read_csv("raw_data/olist_orders_dataset.csv")
order_items = pd.read_csv("raw_data/olist_order_items_dataset.csv")
payments = pd.read_csv("raw_data/olist_order_payments_dataset.csv")
reviews = pd.read_csv("raw_data/olist_order_reviews_dataset.csv")
products = pd.read_csv("raw_data/olist_products_dataset.csv")
sellers = pd.read_csv("raw_data/olist_sellers_dataset.csv")
category_translation = pd.read_csv("raw_data/product_category_name_translation.csv")

for name,df in[("customers",customers),("orders",orders),
("order_items",order_items),("payments",payments),
("reviews",reviews),("products",products),
("sellers",sellers)]:
 print(f"{name}: {df.shape[0]} rows,{df.shape[1]} columns")

customers: 99441 rows,5 columns
orders: 99441 rows,8 columns
order_items: 112650 rows,7 columns
payments: 103886 rows,5 columns
reviews: 99224 rows,7 columns
products: 32951 rows,9 columns
sellers: 3095 rows,4 columns


Inspect for Missing values and Duplicates

In [3]:
tables={
    "customers":customers,
    "orders":orders,
    "order_items":order_items,
    "payments":payments,
    "reviews":reviews,
    "products":products,
    "sellers":sellers
} 
for name, df in tables.items():
    print(f"\nMissing values in {name}:")
    print(df.isnull().sum())
    print(f"Duplicate rows in {name}: {df.duplicated().sum()}")


Missing values in customers:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
Duplicate rows in customers: 0

Missing values in orders:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
Duplicate rows in orders: 0

Missing values in order_items:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
Duplicate rows in order_items: 0

Missing values in payments:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value     

In [4]:
# Drop missing product_category_name from products table
products = products.dropna(subset=["product_category_name"])

Fix Data Types(Dates)

In [5]:
#Orders table Dates
order_date_columns=["order_purchase_timestamp",
              "order_approved_at",
              "order_delivered_carrier_date",
              "order_delivered_customer_date",
              "order_estimated_delivery_date"
              ]
for cols in order_date_columns:
    orders[cols]=pd.to_datetime(orders[cols])

# Order items table dates
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

# Reviews table dates
review_date_columns = ["review_creation_date", "review_answer_timestamp"]
for col in review_date_columns:
    reviews[col] = pd.to_datetime(reviews[col])

print(orders.dtypes)
print(order_items.dtypes)
print(reviews.dtypes)    

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object
order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object
review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


Calculate Delivery Time and Seller Shipping Performance

In [6]:
# 1. How many days did the order take to actually reach the customer?
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

# 2. Was the order delivered before or after the estimated delivery date?
orders["delivery_vs_estimate"] = (
    orders["order_estimated_delivery_date"] - orders["order_delivered_customer_date"]
).dt.days

# 3. Did the seller ship on time? (shipping_limit_date vs actual carrier handoff)
order_items_with_carrier = order_items.merge(
    orders[["order_id", "order_delivered_carrier_date"]], on="order_id", how="left"
)
order_items_with_carrier["seller_shipped_late"] = (
    order_items_with_carrier["order_delivered_carrier_date"] > order_items_with_carrier["shipping_limit_date"]
)

orders[["order_purchase_timestamp", "order_delivered_customer_date", "delivery_days", "delivery_vs_estimate"]].head()
order_items_with_carrier[["shipping_limit_date", "order_delivered_carrier_date", "seller_shipped_late"]].head()

,shipping_limit_date,order_delivered_carrier_date,seller_shipped_late
0,2017-09-19 09:45:35,2017-09-19 18:34:16,True
1,2017-05-03 11:05:13,2017-05-04 14:35:00,True
2,2018-01-18 14:48:30,2018-01-16 12:36:48,False
3,2018-08-15 10:10:18,2018-08-10 13:28:00,False
4,2017-02-13 13:57:51,2017-02-16 09:46:09,True


In [7]:
orders[["order_purchase_timestamp", "order_delivered_customer_date", "delivery_days", "delivery_vs_estimate"]].head()

,order_purchase_timestamp,order_delivered_customer_date,delivery_days,delivery_vs_estimate
0,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0,7.0
1,2018-07-24 20:41:37,2018-08-07 15:27:45,13.0,5.0
2,2018-08-08 08:38:49,2018-08-17 18:06:29,9.0,17.0
3,2017-11-18 19:28:06,2017-12-02 00:28:42,13.0,12.0
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2.0,9.0


In [8]:
order_items_with_carrier[["shipping_limit_date", "order_delivered_carrier_date", "seller_shipped_late"]].head()

,shipping_limit_date,order_delivered_carrier_date,seller_shipped_late
0,2017-09-19 09:45:35,2017-09-19 18:34:16,True
1,2017-05-03 11:05:13,2017-05-04 14:35:00,True
2,2018-01-18 14:48:30,2018-01-16 12:36:48,False
3,2018-08-15 10:10:18,2018-08-10 13:28:00,False
4,2017-02-13 13:57:51,2017-02-16 09:46:09,True


Translate Product Categories to English


In [9]:
products = products.merge(category_translation, on="product_category_name", how="left")

In [10]:
products[["product_category_name", "product_category_name_english"]].head()

,product_category_name,product_category_name_english
0,perfumaria,perfumery
1,artes,art
2,esporte_lazer,sports_leisure
3,bebes,baby
4,utilidades_domesticas,housewares


Build Clean, Joined Tables

In [11]:
# Table 1: Orders — enriched with customer info
orders_clean = orders.merge(customers, on="customer_id", how="left")

# Table 2: Order Items — enriched with product category (English), seller info,
# and carrier/seller-shipping-performance info
order_items_clean = (
    order_items_with_carrier
    .merge(products[["product_id", "product_category_name_english"]], on="product_id", how="left")
    .merge(sellers, on="seller_id", how="left")
)

# Table 3: Payments — no changes needed, just a clean copy
payments_clean = payments.copy()

# Table 4: Reviews — drop the free-text comment columns
reviews_clean = reviews.drop(columns=["review_comment_title", "review_comment_message"])

print(orders_clean.shape)
print(order_items_clean.shape)
print(payments_clean.shape)
print(reviews_clean.shape)

order_items_clean.head()

(99441, 14)
(112650, 13)
(103886, 5)
(99224, 5)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,order_delivered_carrier_date,seller_shipped_late,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,2017-09-19 18:34:16,True,cool_stuff,27277,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,2017-05-04 14:35:00,True,pet_shop,3471,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,2018-01-16 12:36:48,False,furniture_decor,37564,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,2018-08-10 13:28:00,False,perfumery,14403,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,2017-02-16 09:46:09,True,garden_tools,87900,loanda,PR


In [12]:
print(orders_clean.shape)
print(order_items_clean.shape)
print(payments_clean.shape)
print(reviews_clean.shape)

order_items_clean.head()

(99441, 14)
(112650, 13)
(103886, 5)
(99224, 5)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,order_delivered_carrier_date,seller_shipped_late,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,2017-09-19 18:34:16,True,cool_stuff,27277,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,2017-05-04 14:35:00,True,pet_shop,3471,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,2018-01-16 12:36:48,False,furniture_decor,37564,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,2018-08-10 13:28:00,False,perfumery,14403,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,2017-02-16 09:46:09,True,garden_tools,87900,loanda,PR
